[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/22_conv2d_interview.ipynb)

# 🟠 Solution: 2D Convolution（面试版）

## 📋 题目描述

**难度：** Medium

从零实现二维卷积。

卷积将卷积核在二维输入上滑动，在每个位置计算加权和，是 CNN 的基本操作。

**签名:** `my_conv2d(x, weight, bias=None, stride=1, padding=0) -> Tensor`

**参数:**
- `x` — 输入张量 (B, C_in, H, W)
- `weight` — 卷积核张量 (C_out, C_in, kH, kW)
- `bias` — 可选偏置 (C_out,)
- `stride`, `padding` — 整数步幅和零填充

**返回:** 卷积输出张量

**约束:**
- 必须与 `F.conv2d` 数值一致
- 支持 stride 和 padding 参数

**提示：** 1. 手动对输入进行零填充（不用 F.pad）
2. `x.unfold(2, kH, stride).unfold(3, kW, stride)` → 块 `(B, C, H_out, W_out, kH, kW)`
3. `(patches * weight).sum((-3,-2,-1)) + bias`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # ---- Step 1: 零填充 ----
    if padding > 0:
        x = F.pad(x, [padding] * 4)

    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape
    H_out = (H - kH) // stride + 1
    W_out = (W - kW) // stride + 1

    # ---- Step 2: 提取滑动窗口 ----
    # unfold 提取所有位置的 patch，避免显式循环
    # 最终 shape: (B, C_in, H_out, W_out, kH, kW)
    patches = x.unfold(2, kH, stride).unfold(3, kW, stride)

    # ---- Step 3: 手写卷积（不用 einsum）----
    # 面试中可能要求用基础操作手写
    # 方法：patches 与 weight 做点积
    #
    # patches:  (B, C_in, H_out, W_out, kH, kW)
    # weight:   (C_out, C_in, kH, kW)
    # 需要对 C_in, kH, kW 维度求和
    #
    # 用 view + matmul：
    # patches reshape → (B * H_out * W_out, C_in * kH * kW)
    # weight  reshape → (C_out, C_in * kH * kW)
    # matmul → (B * H_out * W_out, C_out)
    patches_flat = patches.reshape(B, H_out, W_out, -1)  # (B, H_out, W_out, C_in*kH*kW)
    weight_flat = weight.reshape(C_out, -1)  # (C_out, C_in*kH*kW)
    # 转置后做矩阵乘法
    out = torch.matmul(patches_flat, weight_flat.T)  # (B, H_out, W_out, C_out)
    out = out.permute(0, 3, 1, 2)  # (B, C_out, H_out, W_out)

    if bias is not None:
        out = out + bias.view(1, -1, 1, 1)
    return out

# 面试常见追问：
# Q: unfold 和循环的区别？
# A: unfold 是视图操作（零拷贝），比显式循环高效得多
# Q: im2col 是什么？
# A: 将卷积转化为矩阵乘法——把每个 patch 展开为列向量
#    unfold + reshape 本质就是 im2col
# Q: stride > 1 时输出尺寸怎么算？
# A: H_out = floor((H - kH) / stride) + 1


In [ ]:
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
b = torch.randn(16)
out = my_conv2d(x, w, b, stride=1, padding=1)
ref = F.conv2d(x, w, b, stride=1, padding=1)
print(f'Output: {out.shape}, Match: {torch.allclose(out, ref, atol=1e-5)}')


In [ ]:
from torch_judge import check
check('conv2d')


## 📝 核心思路总结

1. **unfold 提取窗口**：在 H/W 维度上提取滑动窗口，得到所有 patch
2. **einsum/matrox 乘法**：patch 与卷积核做点积，对 C_in×kH×kW 求和
3. **输出尺寸公式**：`H_out = (H - kH) // stride + 1`
4. **等价于 im2col**：unfold + reshape 就是经典的 im2col 方法
